# Task
Create a complete pipeline for multilabel classification of clinical text using `emilyalsentzer/Bio_ClinicalBERT`. This includes:
1.  Generating a synthetic dataset of 500 clinical records with EHR text and guideline-based screening targets (Breast, Colon, Lung, Prostate, Cervical).
2.  Fine-tuning the BERT model on this data.
3.  Evaluating performance with Precision, Recall, and F1-score.
4.  Saving the dataset, trained model, and a `README.md` documentation file to Google Drive.

## Environment Setup

### Subtask:
Mount Google Drive and install necessary Python libraries for deep learning and NLP.


**Reasoning**:
Mount Google Drive to the Colab environment to enable persistent storage for models and datasets.



In [ ]:
!pip install transformers datasets accelerate scikit-learn

## Generate Synthetic Clinical Data

### Subtask:
Create a synthetic dataset of 500 clinical records with demographics, text notes, and screening labels.


In [ ]:
import pandas as pd
import numpy as np
import random

# Set seed for reproducibility
random.seed(42)
np.random.seed(42)

def generate_synthetic_data(num_records=2000):
    data = []

    genders = ['Male', 'Female']
    smoking_statuses = ['Current', 'Former', 'Never']
    chief_complaints = [
        'Annual physical exam',
        'Persistent cough',
        'Lower back pain',
        'Fatigue and weakness',
        'Routine checkup',
        'Shortness of breath',
        'Abdominal pain',
        'Headache'
    ]

    for _ in range(num_records):
        # 1. Random Demographics
        age = random.randint(30, 80)
        gender = random.choice(genders)
        smoking_status = random.choice(smoking_statuses)
        chief_complaint = random.choice(chief_complaints)

        # 2. Construct Clinical Text
        # Template: [Age]yo [Gender] presents for [Chief Complaint]. [History]. Smoking status: [Smoking Status].

        history_templates = [
            "Patient reports no significant past medical history.",
            "Patient has a history of hypertension.",
            "History notable for type 2 diabetes.",
            "Patient denies recent illness but feels generally unwell.",
            "Symptoms have been present for approximately 2 weeks."
        ]
        history = random.choice(history_templates)

        text_note = (f"{age}yo {gender} presents for {chief_complaint}. "
                     f"{history} "
                     f"Patient is a {smoking_status.lower()} smoker. "
                     f"Review of systems is otherwise negative.")

        # 3. Rule-based Logic for Screening Labels
        # Breast: Female and Age >= 40
        label_breast = 1 if (gender == 'Female' and age >= 40) else 0

        # Cervical: Female and Age between 21 and 65 (inclusive)
        label_cervical = 1 if (gender == 'Female' and 21 <= age <= 65) else 0

        # Colon: Age >= 45
        label_colon = 1 if (age >= 45) else 0

        # Prostate: Male and Age >= 50
        label_prostate = 1 if (gender == 'Male' and age >= 50) else 0

        # Lung: Age >= 50 AND Smoking_Status is 'Current' or 'Former'
        label_lung = 1 if (age >= 50 and smoking_status in ['Current', 'Former']) else 0

        data.append({
            'age': age,
            'gender': gender,
            'smoking_status': smoking_status,
            'text': text_note,
            'Breast': label_breast,
            'Cervical': label_cervical,
            'Colon': label_colon,
            'Prostate': label_prostate,
            'Lung': label_lung
        })

    return pd.DataFrame(data)

# Generate the dataset
df = generate_synthetic_data(500)

# Display first 5 rows
print("First 5 rows of the synthetic dataset:")
print(df.head())

# Verify data distribution
print("\nLabel Distribution (Count of 1s):")
print(df[['Breast', 'Cervical', 'Colon', 'Prostate', 'Lung']].sum())

## Data Preparation and Tokenization

### Subtask:
Prepare the data for training by formatting labels, splitting into train/test sets, converting to Hugging Face Datasets, and tokenizing the text.


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

# 1. Define target label columns
label_cols = ['Breast', 'Cervical', 'Colon', 'Lung', 'Prostate']

# 2. Create 'labels' column (list of floats)
df['labels'] = df[label_cols].values.tolist()
df['labels'] = df['labels'].apply(lambda x: [float(i) for i in x])

# 3. Split into train/test sets (80/20)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# 4. Convert to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# 5. Load Tokenizer
model_checkpoint = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 6. Define preprocessing function
def preprocess_function(examples):
    # Tokenize the text
    encoding = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    # Add labels
    encoding["labels"] = examples["labels"]
    return encoding

# 7. Apply preprocessing
encoded_train_dataset = train_dataset.map(preprocess_function, batched=True)
encoded_test_dataset = test_dataset.map(preprocess_function, batched=True)

# 8. Verify the first example
print("First example keys:", encoded_train_dataset[0].keys())
print("First example labels:", encoded_train_dataset[0]['labels'])

## Model Initialization

### Subtask:
Initialize the Bio_ClinicalBERT model for multilabel classification.


In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

# Define label mappings
labels = ['Breast', 'Cervical', 'Colon', 'Lung', 'Prostate']
id2label = {idx: label for idx, label in enumerate(labels)}
label2id = {label: idx for idx, label in enumerate(labels)}

# Load Model
model = AutoModelForSequenceClassification.from_pretrained(
    "emilyalsentzer/Bio_ClinicalBERT",
    num_labels=5,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model loaded and moved to {device}")

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import torch
import os

# 1. Define Training Arguments
# Note: 'logging_dir' is deprecated, so we set TENSORBOARD_LOGGING_DIR env var or ignore if not using tensorboard explicitly here.
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    eval_strategy="epoch",  # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=10,
)

# 2. Define Metrics Function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Apply sigmoid to logits
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    # Use threshold 0.5 for binary classification per label
    y_pred = np.zeros(probs.shape)
    y_pred[probs >= 0.5] = 1

    # Compute metrics
    y_true = labels
    f1_micro = f1_score(y_true, y_pred, average='micro')
    f1_macro = f1_score(y_true, y_pred, average='macro')
    accuracy = accuracy_score(y_true, y_pred)

    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'accuracy': accuracy
    }

# 3. Initialize Data Collator
# Explicitly handle padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_test_dataset,
    data_collator=data_collator, # Pass data_collator instead of tokenizer
    compute_metrics=compute_metrics,
)

print("Trainer initialized successfully.")

## Train and Evaluate

### Subtask:
Train the Bio_ClinicalBERT model using the initialized Trainer and evaluate its performance on the test set.


In [ ]:
# 1. Train the model
print("Starting training...")
trainer.train()

# 2. Evaluate the model
print("Starting evaluation...")
eval_results = trainer.evaluate()

# 3. Print evaluation metrics
print("\nEvaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

In [ ]:
from sklearn.metrics import classification_report

# Iterate through each label and print a specific classification report
print("Individual Classification Reports per Label:\n")
for i, label in enumerate(label_cols):
    print(f"--- Report for {label} ---")
    # Generate binary report for the specific label column
    print(classification_report(y_true[:, i], y_pred[:, i], target_names=['Negative', 'Positive']))
    print("\n")

## Evaluate Model Performance

### Subtask:
Evaluate the model on the test set and generate a detailed classification report for each screening label.


In [ ]:
from sklearn.metrics import classification_report

# 1. Get predictions
predictions = trainer.predict(encoded_test_dataset)

# 2. Process logits
# Apply sigmoid to logits
sigmoid = torch.nn.Sigmoid()
probs = sigmoid(torch.Tensor(predictions.predictions))

# Threshold at 0.5
y_pred = np.zeros(probs.shape)
y_pred[probs >= 0.5] = 1

# 3. Get true labels from the test dataframe
y_true = test_df[label_cols].values

# 4. Generate Classification Report
report = classification_report(
    y_true,
    y_pred,
    target_names=label_cols,
    zero_division=0
)

print("Detailed Classification Report:")
print(report)